# 🏠 Airbnb NYC 2019 – Sentiment Analysis
## Notebook 3: Deep Learning – Bidirectional LSTM

**Pipeline:**
1. Tokenize text with Keras Tokenizer
2. Pad sequences to fixed length
3. Build Embedding → BiLSTM → Dense architecture
4. Train with EarlyStopping & ReduceLROnPlateau
5. Plot accuracy / loss curves
6. Save model

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

BASE    = Path('.')
CLEANED = BASE / 'Dataset' / 'Cleaned Data'
MODELS  = BASE / 'Models'

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': '#0f0f1a',
                     'axes.facecolor': '#1a1a2e', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'text.color': 'white', 'legend.facecolor': '#1a1a2e'})
print(f'TensorFlow {tf.__version__}')

In [ ]:
df = pd.read_csv(CLEANED / 'reviews_cleaned.csv')
df.dropna(subset=['cleaned_text', 'sentiment'], inplace=True)

le = LabelEncoder()
y  = le.fit_transform(df['sentiment'])  # Negative=0, Neutral=1, Positive=2
y_cat = keras.utils.to_categorical(y, num_classes=3)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'].values, y_cat, test_size=0.2,
    random_state=42, stratify=y
)
print(f'Train: {len(X_train_raw):,}  |  Test: {len(X_test_raw):,}')

In [ ]:
VOCAB_SIZE  = 20_000
MAX_LEN     = 120
EMBED_DIM   = 128

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = encode(X_train_raw)
X_test  = encode(X_test_raw)
print(f'Sequences padded to length {MAX_LEN}')

In [ ]:
def build_bilstm(vocab_size, embed_dim, seq_len, num_classes=3):
    inputs = keras.Input(shape=(seq_len,), name='input')
    x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)
    model = keras.Model(inputs, outputs)
    return model

model = build_bilstm(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True,
                  verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── Training Curves (Plotly) ──────────────────────────────────────────────────
hist = history.history
epochs_ran = list(range(1, len(hist['accuracy']) + 1))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Accuracy', 'Loss'])

for label, key, col in [('Train Acc', 'accuracy', 1), ('Val Acc', 'val_accuracy', 1),
                         ('Train Loss', 'loss', 2), ('Val Loss', 'val_loss', 2)]:
    fig.add_trace(
        go.Scatter(x=epochs_ran, y=hist[key], name=label,
                   mode='lines+markers'),
        row=1, col=col
    )

fig.update_layout(title='<b>BiLSTM Training Curves</b>',
                  template='plotly_dark', title_font_size=18)
fig.show()

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {loss:.4f}  |  Test Accuracy: {acc:.4f}')

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig = px.imshow(
    cm, text_auto=True, x=le.classes_, y=le.classes_,
    color_continuous_scale='Blues',
    title='<b>Confusion Matrix – BiLSTM</b>', template='plotly_dark'
)
fig.update_layout(xaxis_title='Predicted', yaxis_title='Actual')
fig.show()

In [ ]:
model.save(MODELS / 'bilstm_model.keras')
print('✅ BiLSTM model saved to Models/bilstm_model.keras')